# Building Your First AI Agent with LangChain: From Chatbot to Agent

In this tutorial, we'll explore the evolution from simple chatbots to powerful AI agents by building a pet gift finder. You'll understand the key differences and learn when to use each approach.

## Chatbots vs AI Agents: What's the Difference?

**Chatbots** are conversational interfaces that respond to user input based on their training data. They're great for:
- Answering questions from existing knowledge
- Having conversations
- Providing explanations and advice

**AI Agents** go beyond conversation - they can take actions in the real world using tools. They can:
- Search the web for current information
- Make API calls to external services
- Perform calculations and data analysis
- Execute code and interact with databases

## What You'll Learn
- Start with a simple model (chatbot approach)
- Identify limitations of knowledge-only responses
- Build custom tools for real-world capabilities
- Create a full AI agent with LangChain
- Add multimodal capabilities (text + images)
- Deploy to LangSmith for interactive use

## Prerequisites
- Basic Python knowledge
- OpenAI API key
- Tavily API key (for web search)
- LangSmith API key (optional, for deployment)

Let's start simple and build up!

## Step 1: Environment Setup

First, we'll load our environment variables and optionally enable LangSmith tracing.

In [21]:
from dotenv import load_dotenv
import os

load_dotenv()

# Optional: Enable LangSmith tracing for debugging and monitoring
# Uncomment these lines if you have LangSmith set up
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "pet-gift-finder-tutorial"

print("Environment loaded! LangSmith tracing available if configured.")

Environment loaded! LangSmith tracing available if configured.


## Step 2: Starting Simple - The Chatbot Approach

Let's begin with a basic language model to understand what chatbots can and cannot do.

In [22]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

# Initialize a basic chat model
model = init_chat_model(model="gpt-5-nano")

# Create a system message to define the chatbot's role
system_message = SystemMessage(content="""
You are a helpful pet gift advisor. Help users find Christmas gifts for their pets 
based on your knowledge of pet products and care. Provide specific product suggestions 
when possible.
""")

print("Basic Pet Gift Chatbot initialized!")

Basic Pet Gift Chatbot initialized!


### Testing the Basic Chatbot

In [23]:
# Test the basic chatbot
user_question = HumanMessage(content="I have a playful orange tabby cat. What Christmas gifts would be perfect for him?")

response = model.invoke([system_message, user_question])
print("Chatbot Response:")
print(response.content)

Chatbot Response:
Sounds like a fun plan! A playful orange tabby will love a mix of chase toys, enrichment puzzles, and cozy spots. Here are some gift ideas that tend to be big hits with spirited cats, plus a few safe picks you can grab for Christmas.

Top picks for chase and play
- GoCat Da Bird wand teaser (feather on a string with a long rod) — a classic that sparks a thrilling chase.
- Cat Dancer 101 Cat Toy — simple, durable wire with a cardboard/disk lure; great for interactive play.
- KONG Wobbler treat-dispensing toy — durable and unpredictable movement that keeps him busy.

Smart enrichment and puzzle toys
- Catit Senses 2.0 Play Circuit — a modular ball track that engages a cat’s hunting instincts and nose-work.
- Cat Amazing Interactive Puzzle Toy for Cats — Treat maze that challenges him to figure out how to get the treats out.
- Nina Ottosson by Outward Hound Cat Puzzle Toy — if you’ve seen their cat-specific puzzles, this is a good brain-teaser option.
Tip: rotate puzzle 

### The Limitations Become Clear

Let's ask for something that requires current information:

In [24]:
# Ask for current pricing and availability
current_info_question = HumanMessage(content="""
What are the current prices for interactive cat toys on Amazon? 
Which ones are in stock right now and have good reviews?
""")

response = model.invoke([system_message, current_info_question])
print("Chatbot Response to Current Info Request:")
print(response.content)
print("\n" + "="*50)
print("LIMITATION: The chatbot can't access real-time information!")
print("It can only work with its training data, which has a cutoff date.")

Chatbot Response to Current Info Request:
I can’t fetch real-time Amazon prices, stock status, or live review data. But I can help you quickly find good options and interpret what to look for. If you’d like, I can also tailor a short list once you tell me your region and budget.

How to quickly check current prices, stock, and reviews on Amazon
- Go to Amazon and search:
  - “interactive cat toy”
  - “automatic laser cat toy” (for laser toys)
  - “cat puzzle feeder” (for puzzle/feeding toys)
- Use filters:
  - In Stock
  - Prime eligible (if you want fast shipping)
  - Customer reviews: 4 stars & up
- Sort by:
  - Avg. Customer Review (to see top-rated first)
  - Price (to fit your budget)
- Check each listing for:
  - “In stock” status and shipping time (often shows “Usually ships in 1–2 days”)
  - Number of reviews and overall rating (e.g., 4.5 stars with 1,000+ reviews is a good sign)
  - Verify it’s the product you want (some listings are bundles or non-interactive)

Commonly well-

## Step 3: Why We Need Agents

The chatbot approach has several limitations:

1. **No real-time information** - Can't check current prices, availability, or reviews
2. **No verification** - Can't confirm if products actually exist
3. **Static knowledge** - Information becomes outdated
4. **No actions** - Can't actually help you buy anything

**This is where AI agents shine!** Agents can use tools to:
- Search the web for current information
- Check real prices and availability
- Find the latest products and reviews
- Even help with purchasing (with the right integrations)

Let's build an agent that can actually help!

## Step 4: Creating Tools - The Agent's Superpowers

Now we'll give our AI the ability to take actions in the real world. Tools are Python functions that agents can call to perform specific tasks.

In [25]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

# Initialize the Tavily client for web searching
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for current information about pet gifts, products, and shopping"""
    return tavily_client.search(query)

@tool
def pet_gift_search(pet_type: str, pet_characteristics: str, budget: str = "moderate", location: str = "US") -> Dict[str, Any]:
    """Search for specific pet gifts based on type, characteristics, budget, and location"""
    search_query = f"Christmas gifts for {pet_type} {pet_characteristics} {budget} budget 2024 where to buy {location}"
    return tavily_client.search(search_query)

@tool
def local_store_search(product_name: str, location: str) -> Dict[str, Any]:
    """Find local pet stores and retailers in a specific location that might carry a product"""
    search_query = f"pet stores near {location} {product_name} in stock local retailers"
    return tavily_client.search(search_query)

print("Location-aware tools created! Our agent can now find gifts globally and locate nearby stores.")

Location-aware tools created! Our agent can now find gifts globally and locate nearby stores.


### Testing Our Tools

Let's test one of our tools directly to see how it works:

In [26]:
# Test our tool directly
test_result = web_search.invoke({"query": "best interactive cat toys 2024 Amazon"})
print("Direct tool result (first result):")
print(f"Title: {test_result['results'][0]['title']}")
print(f"URL: {test_result['results'][0]['url']}")
print(f"Content: {test_result['results'][0]['content'][:200]}...")
print("\nThis is real, current information from the web!")

Direct tool result (first result):
Title: Cat Expert Reviews Bestselling Cat Toys on Amazon…And One is ...
URL: https://www.youtube.com/watch?v=35raa_A3MFc
Content: Cat Expert Reviews Bestselling Cat Toys on Amazon…And One is Actually DANGEROUS!🙀
Jackson Galaxy
2570000 subscribers
23213 likes
566684 views
22 Nov 2024
Join me as I review EVEN MORE best-selling cat...

This is real, current information from the web!


## Step 5: Designing the Agent's System Prompt

The system prompt is crucial for agents - it needs to explain not just the role, but also how to use tools effectively.

In [27]:
system_prompt = """
You are a helpful pet gift advisor with access to real-time web search capabilities.

Your role:
- Help pet owners find appropriate Christmas gifts based on their pet's characteristics
- Use your web search tools to find current products, prices, and availability
- Consider pet safety, size, age, and personality when making recommendations
- Provide specific product suggestions with purchasing information
- Offer options across different budget ranges
- Help users find local stores and retailers in their area

When to use tools:
- Use web_search for general queries about pet products, reviews, or shopping
- Use pet_gift_search when you have specific pet characteristics, budget, and location info
- Use local_store_search to find nearby pet stores that might carry specific products
- Always ask for the user's location if they want local shopping options
- Always search for current information rather than relying on outdated knowledge

Location examples:
- For US users: Search Amazon, Petco, PetSmart, Chewy
- For Philippines users: Search Shopee, Lazada, local pet stores in Manila/Cebu/Davao
- For other countries: Adapt to local e-commerce and pet store chains

When analyzing pets from photos:
- Identify breed characteristics that might influence gift choices
- Estimate size and age if possible
- Note any visible personality traits or energy levels

Always prioritize pet safety and provide real, purchasable products with current pricing and local availability.
"""

print("System prompt created with location-aware instructions!")

System prompt created with location-aware instructions!


## Step 6: Creating the AI Agent

Now we'll combine everything into a powerful AI agent:

In [28]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Create our AI agent with tools
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, pet_gift_search, local_store_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()  # Enables conversation memory
)

print("AI Agent created! It now has location-aware search capabilities.")

AI Agent created! It now has location-aware search capabilities.


## Step 7: Adding Streaming Responses

Before we test our agent, let's add streaming capabilities for a better user experience. Streaming shows responses as they're generated, making the interaction feel more natural and responsive.

In [29]:
def stream_agent_response(message, config, show_thinking=True):
    """Stream the agent's response for better user experience"""
    if show_thinking:
        print("🤖 Agent is thinking and searching...\n")
    
    # Stream the response using proper streaming format
    for token, metadata in agent.stream({"messages": [HumanMessage(content=message)]}, config, stream_mode="messages"):
        # token is a message chunk with token content
        # metadata contains which node produced the token
        if token.content:  # Check if there's actual content
            print(token.content, end="", flush=True)  # Print token
    
    print("\n\n✅ Response complete!")

print("Streaming function created! Now responses will appear in real-time.")

Streaming function created! Now responses will appear in real-time.


## Step 8: Testing the Agent with Streaming

Let's test our agent with streaming responses for a better user experience:

In [30]:
from langchain.messages import HumanMessage

# Configuration for conversation memory
config = {"configurable": {"thread_id": "pet_gift_session_1"}}

# Test with streaming - same question we asked the basic chatbot
print("🐱 Question: I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?\n")

stream_agent_response(
    "I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?",
    config
)

🐱 Question: I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?

🤖 Agent is thinking and searching...

Sounds like a super fun project for your energized orange tabby! Here are gift ideas that match a playful, chase-happy cat who loves to knock things off tables. I’ve grouped them by type and budget, with why they’re a good fit and safety tips.

Top gift ideas for a chase-oriented, knock-happy cat
- Wand and teaser toys (laser-safe fun)
  - Why: Perfect for a quick sprint-and-chase session; great for bedtime “workouts.”
  - Safety tips: Keep laser pointers away from eyes; supervise wand play to avoid loose strings or small parts.
  - Budget range: usually $8–25.
  - Examples to look for: feather or ribbon wands with replaceable wurls of feather or plush; some come with rotating wand tips for extra movement.

- Treat-dispensing and puzzle toys
  - Why: Engages the brain and rewards chasing behavior; slows

### Comparing: Non-Streaming vs Streaming

Let's also show the traditional non-streaming approach for comparison:

In [11]:
# Traditional non-streaming approach (for comparison)
print("📝 Traditional Non-Streaming Response:\n")

response = agent.invoke(
    {"messages": [HumanMessage(content="What are some budget-friendly cat toys under $20?")]},
    config
)

print("AI Agent Response (traditional):")
print(response['messages'][-1].content)
print("\n" + "="*50)
print("Notice: Traditional responses appear all at once after processing is complete.")
print("Streaming responses appear progressively, providing better user experience!")

📝 Traditional Non-Streaming Response:

AI Agent Response (traditional):
Great question—here are budget-friendly cat toys under $20 that are popular with energetic, playful cats like yours. All options are readily available in the US.

- Frisco Bird with Feathers Teaser Wand Cat Toy with Catnip (Blue)
  - Why it suits him: a classic teaser wand that triggers chasing and pouncing, perfect for burning energy in short sessions.
  - Price: about $5.99 (often $5–$6), with autoship around $5.69 at Chewy.
  - Where to buy: Chewy (Frisco Bird Teaser Wand) and similar pet retailers.

- Frisco Winter Sports Bird Teaser Wand Cat Toy
  - Why it suits him: another wand option with a different feather design—great for variety and long bursts of chasing.
  - Price: about $8.59 at Chewy.
  - Where to buy: Chewy.

- CATSTAGES Tower of Tracks Cat Toy
  - Why it suits him: a multi-layer ball track that rolls around, satisfying his chase instincts and knocking a few balls around in a safe way.
  - Price: a

### Testing Current Information with Streaming - The Agent's Superpower

In [12]:
from langchain.messages import HumanMessage

# Configuration for conversation memory
config = {"configurable": {"thread_id": "pet_gift_session_1"}}

# Ask the same question we asked the basic chatbot
response = agent.invoke(
    {"messages": [HumanMessage(content="I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?")]},
    config
)

print("AI Agent Response (with real-time web search):")
print(response['messages'][-1].content)

AI Agent Response (with real-time web search):
Awesome idea to treat a playful orange tabby this Christmas. Here are budget-friendly gift options that fit a cat who loves chasing and knocking things off tables. I organized them by price tier and included why they’re a good fit and where to buy in the US.

Under $10
- Frisco Bird with Feathers Teaser Wand Cat Toy with Catnip (Blue)
  - Why: perfect for quick chase sessions and lots of pounce-action without needing a big setup.
  - Price: around $5–$6; often on sale.
  - Where to buy: Chewy (Frisco Bird Teaser Wand) or similar pet retailers.
- Frisco Winter Sports Bird Teaser Wand Cat Toy
  - Why: another wand design with different feathers for variety.
  - Price: roughly $8–$9.
  - Where to buy: Chewy.
- Replacement wand/feathers for Da Bird wand
  - Why: inexpensive way to refresh a favorite chase toy.
  - Price: usually under $10 for refill packs.
  - Where to buy: Chewy or Amazon.

$10–$20
- CATSTAGES Tower of Tracks Cat Toy
  - Why:

### Testing Current Information - The Agent's Superpower

In [ ]:
# Test current information with streaming - the question that stumped the chatbot
print("💰 Question: What are the current prices for interactive cat toys on Amazon? Which ones are in stock right now and have good reviews?\n")

stream_agent_response(
    "What are the current prices for interactive cat toys on Amazon? Which ones are in stock right now and have good reviews?",
    config
)

print("\n" + "="*50)
print("🎉 SUCCESS: The agent can access real-time information with streaming!")
print("You can see the agent thinking, using tools, and building the response in real-time.")

💰 Question: What are the current prices for interactive cat toys on Amazon? Which ones are in stock right now and have good reviews?

🤖 Agent is thinking and searching...



## Step 9: Location-Aware Shopping with Streaming

Let's test the location-aware capabilities with streaming responses for different regions:

In [14]:
# Example for Philippines users with streaming
print("🇵🇭 Question: I'm in Manila, Philippines and have a small Shih Tzu. What Christmas gifts can I find locally or on Filipino e-commerce sites like Shopee or Lazada? Budget is around 1000-2000 PHP.\n")

stream_agent_response(
    "I'm in Manila, Philippines and have a small Shih Tzu. What Christmas gifts can I find locally or on Filipino e-commerce sites like Shopee or Lazada? Budget is around 1000-2000 PHP.",
    config
)

🇵🇭 Question: I'm in Manila, Philippines and have a small Shih Tzu. What Christmas gifts can I find locally or on Filipino e-commerce sites like Shopee or Lazada? Budget is around 1000-2000 PHP.

🤖 Agent is thinking and searching...



✅ Response complete!


''

In [15]:
# Example for finding local stores with streaming
print("🏪 Question: I found a great interactive puzzle feeder online, but I'd prefer to buy it locally. I'm in Austin, Texas. Can you help me find pet stores nearby that might carry puzzle feeders?\n")

stream_agent_response(
    "I found a great interactive puzzle feeder online, but I'd prefer to buy it locally. I'm in Austin, Texas. Can you help me find pet stores nearby that might carry puzzle feeders?",
    config
)

🏪 Question: I found a great interactive puzzle feeder online, but I'd prefer to buy it locally. I'm in Austin, Texas. Can you help me find pet stores nearby that might carry puzzle feeders?

🤖 Agent is thinking and searching...



✅ Response complete!


''

## Step 10: Advanced Streaming Example

Let's test streaming with a more complex query that will require multiple tool calls:

In [16]:
# Test streaming with a complex query that requires multiple searches
print("🐕 Complex Question: I have a senior dog (12 years old) with arthritis. What Christmas gifts would help with his comfort and mobility? Please include current prices and where to buy them.\n")

stream_agent_response(
    "I have a senior dog (12 years old) with arthritis. What Christmas gifts would help with his comfort and mobility? Please include current prices and where to buy them.",
    config
)

🐕 Complex Question: I have a senior dog (12 years old) with arthritis. What Christmas gifts would help with his comfort and mobility? Please include current prices and where to buy them.

🤖 Agent is thinking and searching...



✅ Response complete!


''

## Step 11: Adding Multimodal Capabilities

Let's add image analysis so users can upload photos of their pets:

In [17]:
from ipywidgets import FileUpload
from IPython.display import display
import base64

uploader = FileUpload(
    accept='.png,.jpg,.jpeg', 
    multiple=False,
    description='Upload Pet Photo'
)
display(uploader)

FileUpload(value=(), accept='.png,.jpg,.jpeg', description='Upload Pet Photo')

In [18]:
def process_uploaded_image():
    """Convert uploaded image to base64 format for the AI model"""
    if not uploader.value:
        return None, None
    
    # Get the uploaded file
    uploaded_file = uploader.value[0]
    
    # Convert memoryview to bytes
    content_mv = uploaded_file["content"]
    img_bytes = bytes(content_mv)
    
    # Base64 encode for the model
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    
    return img_b64, uploaded_file["type"]

# Process the image
img_b64, file_type = process_uploaded_image()

if img_b64:
    print("Image processed successfully and ready for analysis!")
else:
    print("Please upload an image first.")

Please upload an image first.


In [19]:
if img_b64:
    # Create a multimodal message with both text and image
    multimodal_message = HumanMessage(content=[
        {
            "type": "text", 
            "text": "Here's a photo of my pet! Please analyze their appearance, size, breed characteristics, and any personality traits you can observe, then recommend Christmas gifts that would be perfect for them. Include specific products and where to buy them."
        },
        {
            "type": "image", 
            "base64": img_b64, 
            "mime_type": file_type
        }
    ])
    
    # Send to our agent
    response = agent.invoke(
        {"messages": [multimodal_message]},
        config
    )
    
    print(response['messages'][-1].content)
else:
    print("Please upload a pet photo first!")

Please upload a pet photo first!


## Step 12: Monitoring with LangSmith (Optional)

LangSmith provides powerful tracing and debugging capabilities for your agents. If you have LangSmith set up, you can monitor:

- **Tool usage**: See which tools your agent calls and why
- **Performance metrics**: Track response times and token usage
- **Debugging**: Inspect the full reasoning chain when things go wrong
- **User feedback**: Collect ratings and improve your agent over time

To enable LangSmith tracing, uncomment the lines in Step 1 and set your LangSmith API key in your `.env` file:

```
LANGCHAIN_API_KEY=your_langsmith_api_key_here
```

Once enabled, you can view traces at [smith.langchain.com](https://smith.langchain.com)

## Key Learnings: Chatbots vs Agents + Streaming Benefits

Through this tutorial, we've seen the evolution from simple chatbots to powerful AI agents with streaming capabilities:

### Chatbots (Step 2)
- ✅ Good for: Conversations, explanations, advice based on training data
- ❌ Limited by: Static knowledge, no real-time information, can't take actions

### AI Agents (Steps 4-13)
- ✅ Can: Search the web, find current prices, locate nearby stores, process images
- ✅ Provide: Real-time information, location-aware recommendations, actionable results
- ✅ Scale: Add new tools easily, monitor performance, stream responses

### Streaming Benefits (Steps 7-10)
- ✅ **Better UX**: Users see responses as they're generated, not all at once
- ✅ **Transparency**: Shows when tools are being used and what the agent is thinking
- ✅ **Engagement**: Keeps users engaged during longer processing times
- ✅ **Real-time feedback**: Users can see the agent working through complex queries

### When to Use Each
- **Use chatbots** for: FAQ systems, educational content, creative writing
- **Use agents** for: Shopping assistance, research tasks, data analysis, real-world actions
- **Use streaming** for: Any agent interaction where response time > 2-3 seconds

The key insight: **Streaming Agents = Chatbots + Tools + Real-world capabilities + Better UX!**

## Step 13: Deploying to LangSmith

To deploy this agent to LangSmith for interactive use:

1. Make sure your `.env` file has all required API keys
2. Use the provided `langgraph_pet_gift.json` configuration file
3. Deploy with: `langgraph deploy --config langgraph_pet_gift.json`

Your agent will then be available through LangSmith's web interface for interactive chat!